# Transformer From Scratch

Based on Neel Nanda's GPT-2 From Scratch

Adapted to use only HuggingFace and PyTorch, no other dependencies (Neel's EasyTransformers is great, but incompatible with some versions of )
Training vs inference?

TODO add extended desc

TODO put in git repo, organize, publish. see how they fit together

## Reference Model

In [22]:
# We don't use Neel's EasyTransformers since dependencies have changed
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"  # this is gpt2-small

reference_gpt2 = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

reference_gpt2.eval()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9604.78it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## Setup, Utilities, etc

Markdown Syntax
$$
\begin{bmatrix}
a & b \\
c & d
\end{bmatrix}
$$

In [10]:
%pip install transformers
%pip install fancy_einsum
%pip install einops


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import einops
from fancy_einsum import einsum
from dataclasses import dataclass
import torch
import torch.nn as nn
import numpy as np
import math
import tqdm.auto as tqdm

In [ ]:
#Config object to set params
@dataclass
class Config:
    d_model: int = 768 # size of residual string
    debug: bool = True
    layer_norm_epsilon: float = 1e-5 # added in layernorm to prevent division by 0
    
cfg = Config()
print(cfg)

Config(d_model=768, debug=True)


## Shapes of Reference Model

Key:
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 3072 (4 * d_model)
d_head = 64 (d_model / n_heads)

In [ ]:
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = reference_gpt2.to_tokens(reference_text)
print(tokens)
print(tokens.shape)
print(reference_gpt2.to_str_tokens(tokens))
tokens = tokens.cuda()
logits, cache = reference_gpt2.run_with_cache(tokens)
print(logits.shape)

NameError: name 'tokens' is not defined

In [ ]:
# Activation Shapes
for activation_name, activation in cache.cache_dict.items():
    if ".0." in activation_name or "blocks" not in activation_name: # only first block
        print(activation_name, activation.shape)

NameError: name 'cache' is not defined

# LayerNorm

- Applied at the start of each layer (MLP, attention)
- Normalizing function, taking an input vector and transforming it to have mean = 0 and variance = 1
- Also applies some non-linear scaling with the learned weights
- Neel describes this as an area where pretty cool math takes place - elementwise scaling and translation 

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w = nn.Parameter(torch.ones(cfg.d_model)) # norm by default = 1
        self.b = nn.Parameter(torch.zeros(cfg.d_model)) # mean by default = 0
        
    def forward(self, residual):
        # residual: [batch, position, d_model]
        if cfg.debug: print("Residual:", residual.shape)
        # einops.reduce(tensor, pattern, reduction, optional axes length) -> tensor
        residual = residual - einops.reduce("batch position d_model -> batch position", residual, "mean")
        # calculate variance and square root, then add epsilon to ensure non-zero
        #   variance: sum of squares / n
        scale = (einops.reduce("batch position d_model -> batch position", residual.pow(2), "var") + cfg.layer_norm_epsilon).sqrt()
        residual = residual / scale
        normalized = normalized * self.w + self.b
        if cfg.debug: print("Normalized:", residual.shape)
        return normalized

# Embedding

- A lookup table from input token to vector
- Note that the process of tokenizing (raw input language to tokens) uses Byte-Pair Encoding (shoutout CS240E at Waterloo!). We start with only having tokens for individual letters (ie. `a=1, b=2, c=3` - a bit of an oversimplication), and merging the most common groups of sequences to create a vocabulary of short strings (ie. the pair `a` + `b` occurs often, so we create in our dictionary `ab=4`), some of which are complete words and others are not. This forms our vocabulary of tokens, where each string corresponds to an integer.
- We take an input `token_1, token_2, token_3, ...` and convert it into a tensor (matrix) via matrix multiplication with one-hot embedded vectors

`token` (an integer) $\cdot \begin{bmatrix} 0 \\ 0 \\ 0 \\ ... \\ 1 \\ ... \\ 0 \end{bmatrix} $



# Postitional Embedding
- We add information about the position of each token. 
- TODO check: Note that tokens that are close together should have a greater influence on one another - intuitively, in the sentence "the dancer leaped across the stage, and the lively crowd applauded", "lively" impacts the meaning of "crowd" a lot more than "dancer".

# Attention
- This is the only step that moves information from a prior position in the sequence to the current one
- We parallelize this - we do this for each token for prefix token strings
- n heads etc

# MLP

# Unembedding

- Transforms the final vector into a tensor of logits using a softmax function
$ softmax(x_i) = \frac{e^{x_i}}{\sum e^{x_j}} $
- This output is a tensor of logits, where we have one vector of size $d_{vocab}$ for each input token. 

$$
\begin{bmatrix}
a_0 & b_0 & ...\\
a_1 & b_1 & ...\\
... \\
a_{d} & b_d & ...
\end{bmatrix}
$$